# 01 — Getting Started
## 跑通整个流程：拉数据 → 看图 → 计算指标

这个 notebook 是 Phase 0 的交付物。跑完这个，你就完成了：
- ✅ 从 akshare 拉 A 股真实数据
- ✅ 画 OHLCV 图、收益率分布
- ✅ 计算夏普、最大回撤等核心指标
- ✅ 跑通双均线策略的完整回测

In [ ]:
import sys
sys.path.insert(0, "../../..")   # 指向项目根目录，使 utils/strategies/backtest 可导入

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 项目模块
# from utils.data_loader import get_stock_history, get_index_history, calc_returns  # 已改用本地数据
from utils.data_loader import get_index_history, calc_returns
from utils.local_data_loader import load_local_stock
from utils.metrics import performance_summary
from utils.plotting import (
    plot_cumulative_returns,
    plot_drawdown,
    plot_monthly_returns_heatmap,
    plot_returns_distribution,
)

print("✅ 所有模块导入成功")

## 1. 拉数据

用平安银行（000001）做例子。数据来自 akshare，免费无需注册。
第一次会从网络下载并缓存到 `data/raw/`，之后读缓存。

In [ ]:
SYMBOL = "000001"   # 平安银行
START  = "2018-01-01"
END    = "2024-12-31"

# df = get_stock_history(SYMBOL, START, END, adjust="qfq")  # 已改用本地数据
df = load_local_stock(SYMBOL)  # 从本地 CSV 加载
df = df.loc[START:END]         # 截取日期范围
print(f"数据形状: {df.shape}")
print(f"日期范围: {df.index[0].date()} → {df.index[-1].date()}")
df.tail(5)

## 2. 收盘价走势 + 成交量

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(df.index, df["close"], linewidth=1.2, color="steelblue", label="收盘价（前复权）")
ax1.set_title(f"{SYMBOL} 平安银行 — 日线收盘价", fontsize=14, fontweight="bold")
ax1.set_ylabel("价格（元）")
ax1.legend()

ax2.bar(df.index, df["volume"] / 1e8, color="steelblue", alpha=0.5, label="成交量")
ax2.set_ylabel("成交量（亿股）")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.legend()

plt.tight_layout()
plt.show()

## 3. 收益率分析

### 3.1 简单收益率 vs 对数收益率

- **简单收益率**：`r_t = (P_t - P_{t-1}) / P_{t-1}`，直接对应实际盈亏
- **对数收益率**：`r_t = ln(P_t / P_{t-1})`，可加性好，适合统计建模
- 短期内两者非常接近，长期复利用简单收益率更准确

In [ ]:
simple_ret = df["close"].pct_change().dropna()
log_ret    = np.log(df["close"] / df["close"].shift(1)).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(simple_ret, bins=60, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].axvline(simple_ret.mean(), color="red", linestyle="--", label=f"均值 {simple_ret.mean():.3%}")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("简单日收益率分布")
axes[0].set_xlabel("日收益率")
axes[0].legend()

axes[1].hist(log_ret, bins=60, color="teal", alpha=0.7, edgecolor="white")
axes[1].axvline(log_ret.mean(), color="red", linestyle="--", label=f"均值 {log_ret.mean():.3%}")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("对数日收益率分布")
axes[1].set_xlabel("对数收益率")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"简单收益率 — 均值: {simple_ret.mean():.4%}  标准差: {simple_ret.std():.4%}  偏度: {simple_ret.skew():.3f}  峰度: {simple_ret.kurt():.3f}")
print(f"对数收益率 — 均值: {log_ret.mean():.4%}  标准差: {log_ret.std():.4%}  偏度: {log_ret.skew():.3f}  峰度: {log_ret.kurt():.3f}")

## 4. 绩效指标

In [ ]:
ret = calc_returns(df["close"])
print(performance_summary(ret, name="000001 买入持有"))

In [ ]:
plot_cumulative_returns({"000001 买入持有": ret}, title="000001 平安银行 累计净值")
plt.show()

plot_drawdown(ret, title="000001 回撤曲线")
plt.show()

plot_monthly_returns_heatmap(ret)
plt.show()

## 5. 跑通第一个策略：双均线回测

**双均线交叉（MA Cross）**是最经典的趋势跟踪策略：
- 短期均线（20日）上穿长期均线（60日）→ **金叉，买入**
- 短期均线下穿长期均线 → **死叉，清仓**

这个策略在 A 股表现一般，但用来学习回测框架非常合适。

> ⚠️ **注意**：信号在下一个交易日开盘执行（避免未来函数），并扣除双向千分之二的交易成本。

In [ ]:
from strategies.examples.dual_ma import DualMACrossStrategy
from strategies.base import StrategyConfig
from backtest.engine import BacktestEngine

# 配置策略
config = StrategyConfig(
    name="双均线_20_60",
    params={"fast_period": 20, "slow_period": 60},
    commission=0.0002,
    slippage=0.0002,
    initial_capital=1_000_000,
)

strategy = DualMACrossStrategy(config)
benchmark_returns = calc_returns(df["close"])

engine = BacktestEngine(strategy, df, benchmark_returns=benchmark_returns)
report = engine.run()

In [ ]:
# 可视化：策略 vs 基准
engine.plot()

## 6. 信号可视化：看懂均线金叉/死叉

直观地画出价格 + 双均线 + 买卖点，帮助理解策略逻辑。

In [ ]:
FAST, SLOW = 20, 60
close = df["close"]
ma_fast = close.rolling(FAST).mean()
ma_slow = close.rolling(SLOW).mean()

# 找出金叉/死叉日期（信号切换点）
signal = (ma_fast > ma_slow).astype(int)
crosses = signal.diff().dropna()
golden = crosses[crosses == 1].index   # 金叉
death  = crosses[crosses == -1].index  # 死叉

# 只看最近 2 年，图不那么挤
mask = close.index >= "2022-01-01"
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(close[mask].index, close[mask], color="steelblue", linewidth=1, label="收盘价")
ax.plot(ma_fast[mask].index, ma_fast[mask], color="orange", linewidth=1.2, label=f"MA{FAST}")
ax.plot(ma_slow[mask].index, ma_slow[mask], color="red",    linewidth=1.2, label=f"MA{SLOW}")

# 金叉标记（绿色向上三角）
gd = golden[golden >= "2022-01-01"]
ax.scatter(gd, close.loc[gd], marker="^", color="green", zorder=5, s=80, label="金叉（买入）")

# 死叉标记（红色向下三角）
dd = death[death >= "2022-01-01"]
ax.scatter(dd, close.loc[dd], marker="v", color="red", zorder=5, s=80, label="死叉（卖出）")

ax.set_title(f"{SYMBOL} 双均线信号（2022—2024）", fontsize=13, fontweight="bold")
ax.set_ylabel("价格（元）")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

print(f"区间内金叉次数: {len(gd)}，死叉次数: {len(dd)}")

---

## Phase 0 完成 ✅

恭喜！你已经跑通了完整的量化研究流程：

| 步骤 | 内容 | 状态 |
|------|------|------|
| 1. 拉数据 | akshare 获取 A 股真实 OHLCV | ✅ |
| 2. 可视化 | 收盘价走势 + 成交量图 | ✅ |
| 3. 收益率分析 | 简单收益率 vs 对数收益率，分布统计 | ✅ |
| 4. 绩效指标 | 夏普、最大回撤、年化收益等 | ✅ |
| 5. 策略回测 | 双均线策略完整回测 + 对比基准 | ✅ |
| 6. 信号可视化 | 金叉/死叉标注在价格图上 | ✅ |

### 下一步：Phase 1

- `02_statistics_basics.ipynb`：t 检验、相关性、假设检验——你的策略真的有 alpha 吗？
- `03_single_stock_analysis.ipynb`：系统分析任意一只股票

> 双均线策略在 A 股大概率跑输买入持有。为什么？留给 Phase 1 回答。